In [ ]:


imp ort sys
import subprocess

print("Installing a compatible MCP SDK version...")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--quiet", "--upgrade", "--force-reinstall",
        "mcp==1.26.0"
    ],
    check=True
)

print("Compatible MCP SDK installed.")
print("IMPORTANT: Restart the Colab session before rerunning the experiment.")

Installing a compatible MCP SDK version...
Compatible MCP SDK installed.
IMPORTANT: Restart the Colab session before rerunning the experiment.


In [ ]:

# ================================================================
# EXPERIMENT 6: MCP-BASED DEVELOPER AUTOMATION SYSTEM
# Alternative implementation using an Inventory Management project
# Run this entire script in ONE Google Colab cell.
# ================================================================

import sys, subprocess, os, json, time, ast, uuid, asyncio
import logging, statistics, textwrap
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("EXPERIMENT 6: MCP DEVELOPER AUTOMATION SYSTEM")
print("=" * 70)

# 1. INSTALL DEPENDENCIES
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mcp", "pytest"],
    check=True
)
print("\n[1] Dependencies installed.")

# 2. CREATE PROJECT DIRECTORIES
ROOT = Path("/content/mcp_inventory_agent")
PROJECT = ROOT / "inventory_project"
LOG_DIR = ROOT / "logs"
PROJECT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

TRACE_FILE = LOG_DIR / "mcp_traces.jsonl"
LOG_FILE = LOG_DIR / "execution.log"
REPORT_FILE = LOG_DIR / "developer_automation_report.json"

for p in (TRACE_FILE, LOG_FILE, REPORT_FILE):
    if p.exists():
        p.unlink()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler(sys.stdout)
    ],
    force=True
)
logger = logging.getLogger("MCPInventoryAgent")

# 3. CREATE A DIFFERENT SAMPLE PYTHON PROJECT
(PROJECT / "inventory.py").write_text(textwrap.dedent("""
    INVENTORY = {
        "laptop": 10,
        "mouse": 25,
        "keyboard": 15
    }

    def add_stock(product, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be positive")
        INVENTORY[product] = INVENTORY.get(product, 0) + quantity
        return INVENTORY[product]

    def remove_stock(product, quantity):
        if quantity <= 0:
            raise ValueError("Quantity must be positive")
        if product not in INVENTORY:
            raise KeyError("Product not found")
        if INVENTORY[product] < quantity:
            raise ValueError("Insufficient stock")
        INVENTORY[product] -= quantity
        return INVENTORY[product]

    def get_stock(product):
        if product not in INVENTORY:
            raise KeyError("Product not found")
        return INVENTORY[product]

    def low_stock(threshold=10):
        return {
            product: quantity
            for product, quantity in INVENTORY.items()
            if quantity <= threshold
        }

    def inventory_value(prices):
        return sum(
            quantity * prices.get(product, 0)
            for product, quantity in INVENTORY.items()
        )
""").strip() + "\n", encoding="utf-8")

(PROJECT / "app.py").write_text(textwrap.dedent("""
    from inventory import get_stock, low_stock

    def inventory_summary():
        return {
            "laptop_stock": get_stock("laptop"),
            "low_stock_items": low_stock(10)
        }

    if __name__ == "__main__":
        print(inventory_summary())
""").strip() + "\n", encoding="utf-8")

(PROJECT / "test_inventory.py").write_text(textwrap.dedent("""
    import pytest
    import inventory

    def setup_function():
        inventory.INVENTORY.clear()
        inventory.INVENTORY.update({
            "laptop": 10,
            "mouse": 25,
            "keyboard": 15
        })

    def test_add_stock():
        assert inventory.add_stock("laptop", 5) == 15

    def test_remove_stock():
        assert inventory.remove_stock("mouse", 5) == 20

    def test_get_stock():
        assert inventory.get_stock("keyboard") == 15

    def test_low_stock():
        assert inventory.low_stock(10) == {"laptop": 10}

    def test_invalid_quantity():
        with pytest.raises(ValueError):
            inventory.add_stock("laptop", 0)

    def test_insufficient_stock():
        with pytest.raises(ValueError):
            inventory.remove_stock("laptop", 100)
""").strip() + "\n", encoding="utf-8")

(PROJECT / "README.md").write_text(
    "# Inventory Management Project\n"
    "A sample Python application for stock tracking, "
    "stock updates, low-stock detection, and automated testing.\n",
    encoding="utf-8"
)

print("[2] Inventory project created:", sorted(p.name for p in PROJECT.iterdir()))

# 4. CREATE AN MCP SERVER
# File access is restricted to PROJECT.
# Tests can only run on test_*.py files.
SERVER_FILE = ROOT / "mcp_server.py"

SERVER_SOURCE = r'''
import ast
import os
import subprocess
import sys
import time
from pathlib import Path
from mcp.server.fastmcp import FastMCP

PROJECT = Path(os.environ["MCP_PROJECT"]).resolve()
mcp = FastMCP("Inventory Developer Automation Server")

def safe_path(name):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("A relative file path is required")
    p = (PROJECT / name).resolve()
    try:
        p.relative_to(PROJECT)
    except ValueError:
        raise PermissionError("Path outside the project is blocked")
    if not p.is_file():
        raise FileNotFoundError("Project file not found")
    if p.stat().st_size > 200000:
        raise ValueError("File exceeds the 200 KB limit")
    return p

@mcp.tool()
def list_files() -> dict:
    """List files in the sample inventory project."""
    return {
        "status": "success",
        "files": sorted(p.name for p in PROJECT.iterdir() if p.is_file())
    }

@mcp.tool()
def read_file(file_path: str) -> dict:
    """Read a UTF-8 file confined to the sample project."""
    p = safe_path(file_path)
    return {
        "status": "success",
        "file": p.name,
        "content": p.read_text(encoding="utf-8")
    }

@mcp.tool()
def search_code(query: str) -> dict:
    """Search Python files for a case-insensitive text match."""
    if not isinstance(query, str) or not query.strip():
        return {"status": "failed", "error": "Search query is required"}
    matches = []
    for p in sorted(PROJECT.glob("*.py")):
        for line_no, line in enumerate(
            p.read_text(encoding="utf-8").splitlines(), 1
        ):
            if query.lower() in line.lower():
                matches.append({
                    "file": p.name,
                    "line": line_no,
                    "text": line.strip()
                })
    return {"status": "success", "query": query,
            "match_count": len(matches), "matches": matches[:100]}

@mcp.tool()
def analyze_python(file_path: str) -> dict:
    """Analyze Python syntax, functions, classes and imports with AST."""
    p = safe_path(file_path)
    if p.suffix != ".py":
        return {"status": "failed", "error": "Only Python files are supported"}
    try:
        tree = ast.parse(p.read_text(encoding="utf-8"), filename=p.name)
    except SyntaxError as e:
        return {"status": "failed", "error": str(e), "line": e.lineno}

    functions = [
        {"name": n.name, "line": n.lineno}
        for n in ast.walk(tree)
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]
    classes = [
        {"name": n.name, "line": n.lineno}
        for n in ast.walk(tree) if isinstance(n, ast.ClassDef)
    ]
    imports = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            imports.extend(a.name for a in n.names)
        elif isinstance(n, ast.ImportFrom):
            imports.append(n.module or "")
    return {
        "status": "success", "file": p.name,
        "function_count": len(functions), "functions": functions,
        "class_count": len(classes), "classes": classes,
        "imports": sorted(set(imports))
    }

@mcp.tool()
def run_tests(test_file: str = "test_inventory.py",
              timeout_seconds: int = 15) -> dict:
    """Run a project-local test_inventory-style Python test file."""
    p = safe_path(test_file)
    if p.suffix != ".py" or not p.name.startswith("test_"):
        return {"status": "blocked", "error": "Only test_*.py files are allowed"}
    timeout_seconds = max(1, min(int(timeout_seconds), 20))
    start = time.perf_counter()
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pytest", "-q", p.name],
            cwd=str(PROJECT), capture_output=True, text=True,
            timeout=timeout_seconds
        )
        return {
            "status": "success" if result.returncode == 0 else "failed",
            "return_code": result.returncode,
            "elapsed_seconds": round(time.perf_counter() - start, 4),
            "stdout": result.stdout[-4000:],
            "stderr": result.stderr[-2000:]
        }
    except subprocess.TimeoutExpired:
        return {"status": "error", "retryable": True,
                "error": "Test execution timed out"}
    except Exception as e:
        return {"status": "error", "error": str(e)}

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVER_FILE.write_text(SERVER_SOURCE, encoding="utf-8")
print("[3] MCP server source generated.")

# 5. MCP CLIENT, TOOL DISCOVERY, SUPERVISOR AND EXECUTION
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_FILE)],
    env={
        **os.environ,
        "MCP_PROJECT": str(PROJECT),
        "PYTHONUNBUFFERED": "1"
    }
)

MCP_ERRLOG = open(
    str(LOG_DIR / "mcp_server_stderr.log"),
    "a",
    buffering=1,
    encoding="utf-8"
)

POLICY = {
    "list_files": {"risk": "LOW", "allowed": True},
    "read_file": {"risk": "LOW", "allowed": True},
    "search_code": {"risk": "LOW", "allowed": True},
    "analyze_python": {"risk": "LOW", "allowed": True},
    "run_tests": {"risk": "MEDIUM", "allowed": True}
}

def unpack_result(result):
    structured = getattr(result, "structuredContent", None)
    if structured is not None:
        return structured
    parts = []
    for item in getattr(result, "content", []) or []:
        if getattr(item, "text", None):
            try:
                parts.append(json.loads(item.text))
            except Exception:
                parts.append(item.text)
    return parts[0] if len(parts) == 1 else parts

def trace_write(row):
    with open(TRACE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, default=str) + "\n")

def supervisor(plan, discovered_names):
    approved, rejected = [], []
    for task in plan:
        reason = None
        name = task.get("tool")
        args = task.get("arguments", {})
        if name not in discovered_names:
            reason = "Tool was not discovered"
        elif name not in POLICY or not POLICY[name]["allowed"]:
            reason = "Tool blocked by policy"
        elif not isinstance(args, dict):
            reason = "Arguments must be a dictionary"
        elif name == "read_file":
            if not args.get("file_path"):
                reason = "Missing file_path"
            else:
                try:
                    candidate = (PROJECT / args["file_path"]).resolve()
                    candidate.relative_to(PROJECT.resolve())
                    if not candidate.is_file():
                        reason = "File does not exist"
                except Exception:
                    reason = "Path outside project or invalid"
        elif name == "analyze_python":
            if not str(args.get("file_path", "")).endswith(".py"):
                reason = "A Python file is required"
        elif name == "run_tests":
            fn = args.get("test_file", "test_inventory.py")
            if not str(fn).startswith("test_") or not str(fn).endswith(".py"):
                reason = "Only test_*.py files are allowed"
        if reason:
            rejected.append({**task, "rejection_reason": reason})
        else:
            approved.append(task)
    return {"approved": approved, "rejected": rejected}

async def call_traced(session, task, trace_id, max_retries=1):
    name = task["tool"]
    args = task.get("arguments", {})
    task_id = f"TASK-{task['step']:03d}"

    for attempt in range(max_retries + 1):
        start = time.perf_counter()
        try:
            result = await session.call_tool(name, arguments=args)
            payload = unpack_result(result)
            is_error = getattr(result, "isError", False)
            elapsed = round(time.perf_counter() - start, 4)

            status = (
                "error" if is_error else
                payload.get("status", "success")
                if isinstance(payload, dict) else "success"
            )
            row = {
                "trace_id": trace_id, "task_id": task_id,
                "tool": name, "arguments": args,
                "risk": POLICY[name]["risk"], "attempt": attempt + 1,
                "status": status, "execution_time": elapsed,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "error": payload.get("error") if isinstance(payload, dict) else None,
                "result": payload
            }
            trace_write(row)
            logger.info("%s | %s | %s | %.4fs",
                        task_id, name, status, elapsed)

            retryable = isinstance(payload, dict) and payload.get("retryable") is True
            if status in ("success", "blocked") or not retryable or attempt == max_retries:
                return {
                    "step": task["step"], "task_id": task_id,
                    "tool": name, "status": status,
                    "execution_time": elapsed, "attempts": attempt + 1,
                    "result": payload
                }
        except Exception as e:
            elapsed = round(time.perf_counter() - start, 4)
            trace_write({
                "trace_id": trace_id, "task_id": task_id,
                "tool": name, "status": "error",
                "execution_time": elapsed, "error": str(e),
                "timestamp": datetime.now(timezone.utc).isoformat()
            })
            if attempt == max_retries:
                return {
                    "step": task["step"], "task_id": task_id,
                    "tool": name, "status": "error",
                    "execution_time": elapsed, "error": str(e)
                }
            await asyncio.sleep(0.25)

async def run_workflow():
    trace_id = "TRACE-" + uuid.uuid4().hex[:8].upper()
    async with stdio_client(
    SERVER_PARAMS,
    errlog=MCP_ERRLOG
) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            tool_names = [t.name for t in tools_result.tools]

            print("\n[4] Dynamically discovered MCP tools:")
            for tool in tools_result.tools:
                print("  -", tool.name, ":", tool.description)

            # Deterministic planner: no external LLM/API key required.
            plan = [
                {"step": 1, "tool": "list_files",
                 "arguments": {}, "depends_on": []},
                {"step": 2, "tool": "analyze_python",
                 "arguments": {"file_path": "inventory.py"}, "depends_on": []},
                {"step": 3, "tool": "search_code",
                 "arguments": {"query": "def "}, "depends_on": []},
                {"step": 4, "tool": "read_file",
                 "arguments": {"file_path": "README.md"}, "depends_on": []},
                {"step": 5, "tool": "run_tests",
                 "arguments": {"test_file": "test_inventory.py",
                               "timeout_seconds": 15},
                 "depends_on": [2]}
            ]

            print("\n[5] PLANNER OUTPUT")
            print(json.dumps(plan, indent=2))

            approval = supervisor(plan, tool_names)
            print("\n[6] SUPERVISOR APPROVED:",
                  len(approval["approved"]))
            print("SUPERVISOR REJECTED:",
                  json.dumps(approval["rejected"], indent=2))

            # Phase 1: independent read-only tasks execute concurrently.
            independent = [
                t for t in approval["approved"] if t["step"] != 5
            ]
            t0 = time.perf_counter()
            parallel_results = await asyncio.gather(*[
                call_traced(session, t, trace_id) for t in independent
            ])
            parallel_wall = round(time.perf_counter() - t0, 4)

            # Validate analysis before allowing the dependent test task.
            analysis = next(
                (r for r in parallel_results if r["step"] == 2), None
            )
            test_task = next(
                (t for t in approval["approved"] if t["step"] == 5), None
            )
            dependent_results = []
            if test_task and analysis and analysis["status"] == "success":
                # Phase 2: run tests only after successful analysis.
                dependent_results.append(
                    await call_traced(session, test_task, trace_id)
                )
            elif test_task:
                dependent_results.append({
                    "step": 5, "task_id": "TASK-005",
                    "tool": "run_tests", "status": "blocked_dependency",
                    "error": "Python analysis did not succeed"
                })

            results = sorted(
                parallel_results + dependent_results,
                key=lambda r: r["step"]
            )

            print("\n[7] WORKFLOW RESULTS")
            for r in results:
                print(
                    r["task_id"], "|", r["tool"], "|",
                    r["status"], "|", r.get("execution_time", "-"), "seconds"
                )

            # Result validation
            validation = []
            for r in results:
                payload = r.get("result", {})
                valid = r["status"] == "success"
                if r["tool"] == "analyze_python" and isinstance(payload, dict):
                    valid = valid and "function_count" in payload
                if r["tool"] == "run_tests" and isinstance(payload, dict):
                    valid = valid and payload.get("return_code") == 0
                validation.append({
                    "task_id": r["task_id"], "tool": r["tool"],
                    "valid": bool(valid)
                })

            # 8. BENCHMARK THE SAME FOUR INDEPENDENT TASKS.
            bench_tasks = [
                t for t in independent if t["step"] in (1, 2, 3, 4)
            ]
            sequential_times, parallel_times = [], []
            repeats = 5

            for _ in range(repeats):
                start = time.perf_counter()
                for t in bench_tasks:
                    await call_traced(session, t, trace_id)
                sequential_times.append(time.perf_counter() - start)

                start = time.perf_counter()
                await asyncio.gather(*[
                    call_traced(session, t, trace_id) for t in bench_tasks
                ])
                parallel_times.append(time.perf_counter() - start)

            avg_seq = statistics.mean(sequential_times)
            avg_par = statistics.mean(parallel_times)
            reduction = ((avg_seq - avg_par) / avg_seq * 100) if avg_seq else 0

            performance = {
                "tasks_per_run": len(bench_tasks),
                "repeats": repeats,
                "sequential_seconds": [round(x, 5) for x in sequential_times],
                "parallel_seconds": [round(x, 5) for x in parallel_times],
                "average_sequential_seconds": round(avg_seq, 5),
                "average_parallel_seconds": round(avg_par, 5),
                "relative_time_reduction_percent": round(reduction, 2)
            }

            print("\n[8] PERFORMANCE COMPARISON")
            print(json.dumps(performance, indent=2))

            # 9. SECURITY TESTS: these should be rejected before execution.
            bad_plan = [
                {"step": 91, "tool": "read_file",
                 "arguments": {"file_path": "../../etc/passwd"}},
                {"step": 92, "tool": "run_tests",
                 "arguments": {"test_file": "inventory.py"}},
                {"step": 93, "tool": "unknown_tool", "arguments": {}}
            ]
            security_demo = supervisor(bad_plan, tool_names)
            print("\n[9] EXPECTED SECURITY/POLICY REJECTIONS")
            print(json.dumps(security_demo["rejected"], indent=2))

            report = {
                "experiment": "Experiment 6",
                "title": "MCP Developer Automation - Inventory Project",
                "generated_at": datetime.now(timezone.utc).isoformat(),
                "trace_id": trace_id,
                "project_directory": str(PROJECT),
                "discovered_tools": tool_names,
                "planner": plan,
                "supervision": {
                    "approved_count": len(approval["approved"]),
                    "rejected": approval["rejected"]
                },
                "execution": results,
                "validation": {
                    "items": validation,
                    "overall_valid": all(v["valid"] for v in validation)
                },
                "performance": performance,
                "security_demo": security_demo,
                "artifacts": {
                    "trace": str(TRACE_FILE),
                    "log": str(LOG_FILE),
                    "report": str(REPORT_FILE)
                }
            }

            REPORT_FILE.write_text(
                json.dumps(report, indent=2, default=str),
                encoding="utf-8"
            )

            print("\n[10] FINAL SUMMARY")
            print(json.dumps({
                "trace_id": trace_id,
                "discovered_tools": len(tool_names),
                "workflow_tasks": len(results),
                "successful_tasks": sum(r["status"] == "success" for r in results),
                "failed_or_blocked_tasks": sum(
                    r["status"] != "success" for r in results
                ),
                "all_results_valid": report["validation"]["overall_valid"],
                "average_sequential_seconds": performance["average_sequential_seconds"],
                "average_parallel_seconds": performance["average_parallel_seconds"],
                "performance_reduction_percent": reduction
            }, indent=2))

            print("\n[11] GENERATED ARTIFACTS")
            for p in (LOG_FILE, TRACE_FILE, REPORT_FILE):
                print(f"{p} | exists={p.exists()} | bytes={p.stat().st_size}")

            print("\n[12] SAMPLE TRACE RECORD")
            with open(TRACE_FILE, encoding="utf-8") as f:
                first_line = f.readline()
            print(json.dumps(json.loads(first_line), indent=2)[:2500])

            # Download the report and traces to your computer.
            try:
                from google.colab import files
                for p in (REPORT_FILE, TRACE_FILE, LOG_FILE):
                    files.download(str(p))
            except Exception as e:
                print("Automatic downloads unavailable:", e)
                print("Files remain available in:", LOG_DIR)

            return report

# 10. RUN THE COMPLETE EXPERIMENT
report = await run_workflow()

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED")
print("Project:", PROJECT)
print("Reports:", LOG_DIR)
print("=" * 70)

EXPERIMENT 6: MCP DEVELOPER AUTOMATION SYSTEM

[1] Dependencies installed.
[2] Inventory project created: ['README.md', 'app.py', 'inventory.py', 'test_inventory.py']
[3] MCP server source generated.

[4] Dynamically discovered MCP tools:
  - list_files : List files in the sample inventory project.
  - read_file : Read a UTF-8 file confined to the sample project.
  - search_code : Search Python files for a case-insensitive text match.
  - analyze_python : Analyze Python syntax, functions, classes and imports with AST.
  - run_tests : Run a project-local test_inventory-style Python test file.

[5] PLANNER OUTPUT
[
  {
    "step": 1,
    "tool": "list_files",
    "arguments": {},
    "depends_on": []
  },
  {
    "step": 2,
    "tool": "analyze_python",
    "arguments": {
      "file_path": "inventory.py"
    },
    "depends_on": []
  },
  {
    "step": 3,
    "tool": "search_code",
    "arguments": {
      "query": "def "
    },
    "depends_on": []
  },
  {
    "step": 4,
    "tool": "

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


EXPERIMENT COMPLETED
Project: /content/mcp_inventory_agent/inventory_project
Reports: /content/mcp_inventory_agent/logs
